# Lab 3 · Config, Seed, Logging, Checkpoint

Terkait: **Bab 03 · Eksperimen Reproduksibel**

## Tujuan
1. Memverifikasi reproduksibilitas: dua run dengan seed sama → hasil identik (± 1e-4).
2. Memeriksa metadata checkpoint secara lengkap (epoch, git_hash, config, dirty flag).
3. Memverifikasi resume dari checkpoint: lanjutkan training dari epoch N.
4. Membaca TensorBoard logs dan membandingkan dua run.

## Checklist
- [ ] Dua run dengan seed sama → val_acc identik hingga 4 desimal.
- [ ] Checkpoint berisi commit hash yang valid (bukan `unknown`).
- [ ] Dirty flag terdeteksi benar saat repo punya uncommitted changes.
- [ ] Resume dari checkpoint: training dilanjutkan dari epoch N, bukan epoch 1.
- [ ] Paragraf refleksi tentang trade-off determinism vs kecepatan.

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/muhammad-zainal-muttaqin/ModulePembelajaran.git'
    REPO_DIR = '/content/ModulePembelajaran'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        print('Repo sudah ada, skip clone.')

    %cd {REPO_DIR}/ModulePembelajaran/template_repo
    !pip install -e .
    print('\nSetup Colab selesai. Working dir:', os.getcwd())
else:
    print('Environment lokal. Lewati setup Colab.')

## 0. Setup

### Lokal
```bash
pip install -e .
```

### Google Colab
Jalankan sel kode **0a. Setup Colab** di bawah ini. Setelah selesai, lanjut ke **1. Verifikasi reproduksibilitas**.

## 1. Verifikasi reproduksibilitas

Jalankan dua run dengan seed dan config identik. Hasilnya harus sama persis.

In [ ]:
import subprocess, sys

# Gunakan dry-run untuk cepat (1 epoch, 100 sampel)
# Jalankan dua kali dengan seed sama
for run_id in [1, 2]:
    result = subprocess.run(
        [sys.executable, '-m', 'src.train',
         '--config', 'configs/baseline.yaml',
         '--seed', '42',
         '--output_dir', f'experiments/repro_check',
         '--dry-run'],
        capture_output=True, text=True
    )
    # Ekstrak val_acc dari output
    for line in result.stdout.split('\n'):
        if 'val_acc' in line and 'epoch' in line:
            print(f'Run {run_id}: {line.strip()}')
            break

In [ ]:
import json
from pathlib import Path

# Bandingkan summary dari dua run dry-run — cari dua folder terbaru
exp_dirs = sorted(
    [d for d in Path('experiments').glob('baseline_dryrun_seed42*') if d.is_dir()],
    key=lambda d: d.stat().st_mtime
)[-2:]

if len(exp_dirs) >= 2:
    s1 = json.loads((exp_dirs[0] / 'summary.json').read_text())
    s2 = json.loads((exp_dirs[1] / 'summary.json').read_text())
    diff = abs(s1['best_val_acc'] - s2['best_val_acc'])
    print(f'Run 1 val_acc: {s1["best_val_acc"]:.6f}')
    print(f'Run 2 val_acc: {s2["best_val_acc"]:.6f}')
    print(f'Selisih:       {diff:.6f}')
    if diff < 1e-4:
        print('✓ Reproduksibel: selisih < 1e-4')
    else:
        print('⚠ Tidak reproduksibel! Cek apakah cudnn.deterministic=True di utils.py')
else:
    print('Belum ada 2 run untuk dibandingkan. Jalankan sel di atas dua kali.')

## 2. Inspeksi checkpoint metadata

In [ ]:
import torch

# Muat checkpoint dari training penuh Lab 1 (atau dry-run)
ckpt_path = Path('experiments/baseline_seed42/ckpt_best.pt')
if not ckpt_path.exists():
    # Fallback ke dry-run jika training penuh belum ada
    candidates = sorted(Path('experiments').glob('baseline*seed42*/ckpt_best.pt'))
    ckpt_path = candidates[0] if candidates else None

if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print('=== Checkpoint Metadata ===')
    for k, v in ckpt['meta'].items():
        print(f'  {k:20s}: {v}')
    print()
    print('=== Kunci dalam checkpoint ===')
    print(' ', list(ckpt.keys()))
    print()
    print('=== Config dalam checkpoint (summary) ===')
    for section in ['model', 'loss', 'optim', 'scheduler', 'train']:
        print(f'  {section}: {ckpt["config"].get(section)}')
else:
    print('Checkpoint belum ada. Jalankan Lab 1 dulu.')

## 3. Uji dirty flag

Buat perubahan sementara di repo → jalankan training → cek apakah checkpoint mencatat `dirty=True`.

In [ ]:
import os

# Buat perubahan dummy di file yang tidak penting
dummy_path = Path('experiments/.dirty_test')
dummy_path.write_text('test dirty detection')

# Jalankan dry-run — seharusnya muncul peringatan di log
result = subprocess.run(
    [sys.executable, '-m', 'src.train',
     '--config', 'configs/baseline.yaml', '--seed', '42', '--dry-run'],
    capture_output=True, text=True
)

# Cari peringatan dirty
dirty_warning_found = False
for line in result.stdout.split('\n'):
    if 'dirty' in line.lower():
        print('Output:', line)
        dirty_warning_found = True

if not dirty_warning_found:
    print('(Tidak ada output dirty — mungkin file .dirty_test tidak tracked oleh git)')

# Bersihkan
dummy_path.unlink(missing_ok=True)
print('✓ Dummy file dihapus.')

## 4. Resume dari checkpoint (kritis untuk spot instance)

Training berhenti di tengah jalan? Kita harus bisa melanjutkan dari checkpoint terakhir tanpa mengulang dari epoch 1.

Perhatikan: `train.py` saat ini belum punya flag `--resume`. Di bawah ini adalah contoh kode manual untuk meload checkpoint dan melanjutkan loop — ini exercise untuk memahami mekanismenya.

In [ ]:
from src.models import build_model
from src.utils import load_config, set_seed
from src.losses import build_loss
from src.train import build_optimizer, build_scheduler

cfg = load_config('configs/baseline.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cari checkpoint
candidates = sorted(Path('experiments').glob('baseline*seed42*/ckpt_last.pt'))
if not candidates:
    print('Tidak ada checkpoint ckpt_last.pt. Jalankan training penuh (Lab 1) dulu.')
else:
    ckpt_path = candidates[-1]  # checkpoint terbaru
    ckpt = torch.load(ckpt_path, map_location=device)
    resumed_epoch = ckpt['meta']['epoch']

    # Muat model + optimizer + scheduler ke state tepat saat checkpoint disimpan
    model = build_model(ckpt['config']['model']).to(device)
    model.load_state_dict(ckpt['model_state'])

    total_epochs = ckpt['config']['train']['epochs']
    optimizer = build_optimizer(list(model.parameters()), ckpt['config']['optim'])
    optimizer.load_state_dict(ckpt['optimizer_state'])

    scheduler = build_scheduler(optimizer, ckpt['config'].get('scheduler'), total_epochs)
    if scheduler is not None and ckpt['scheduler_state'] is not None:
        scheduler.load_state_dict(ckpt['scheduler_state'])

    print(f'Checkpoint dimuat: epoch={resumed_epoch}, val_acc={ckpt["meta"]["metric_value"]:.4f}')
    print(f'Siap melanjutkan dari epoch {resumed_epoch + 1} hingga epoch {total_epochs}')
    print(f'LR saat ini: {optimizer.param_groups[0]["lr"]:.6f}')

## 5. Baca dan bandingkan TensorBoard logs

In [ ]:
# Baca val_acc dari dua run berbeda seed dan plot perbandingan
import re
import matplotlib.pyplot as plt

def parse_log(log_path: Path) -> dict:
    """Parse train.log ke dict of lists."""
    data = {'epoch': [], 'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    pattern = re.compile(
        r'epoch=(\d+) train_loss=([\d.]+) train_acc=([\d.]+) val_loss=([\d.]+) val_acc=([\d.]+)'
    )
    for line in log_path.read_text(encoding='utf-8').splitlines():
        m = pattern.search(line)
        if m:
            data['epoch'].append(int(m.group(1)))
            data['train_loss'].append(float(m.group(2)))
            data['train_acc'].append(float(m.group(3)))
            data['val_loss'].append(float(m.group(4)))
            data['val_acc'].append(float(m.group(5)))
    return data

# Cari semua run yang berhasil
run_logs = {}
for log_path in sorted(Path('experiments').glob('baseline_seed*/train.log')):
    seed = log_path.parent.name.split('seed')[-1]
    data = parse_log(log_path)
    if data['epoch']:
        run_logs[f'seed={seed}'] = data

if len(run_logs) < 2:
    print('Perlu ≥2 run untuk perbandingan. Jalankan dengan seed berbeda:')
    print('  Lokal/Notebook Colab: python -m src.train --config configs/baseline.yaml --seed 123')
    print('  Colab terminal: cd /content/ModulePembelajaran/ModulePembelajaran/template_repo && python -m src.train --config configs/baseline.yaml --seed 123')
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ['steelblue', 'tomato', 'seagreen', 'mediumpurple']
    for i, (label, data) in enumerate(run_logs.items()):
        ax.plot(data['epoch'], [v*100 for v in data['val_acc']],
                label=label, color=colors[i % len(colors)])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Accuracy (%)')
    ax.set_title('Perbandingan val_acc antar seed (baseline config)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('experiments/plots/seed_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

    print('\nVariasi antar-seed (val_acc akhir):')
    final_accs = [data['val_acc'][-1] for data in run_logs.values() if data['val_acc']]
    import numpy as np
    print(f'  μ = {np.mean(final_accs)*100:.2f}%,  σ = {np.std(final_accs)*100:.2f}%')
    print(f'  Range: {min(final_accs)*100:.2f}% - {max(final_accs)*100:.2f}%')

## 6. Refleksi

1. Jika kamu menemukan bug dua minggu kemudian dan harus memahami ulang eksperimen ini, apa **tiga minimum** yang kamu butuhkan dari folder eksperimen — bukan dari ingatanmu?

2. `cudnn.deterministic=True` memperlambat training ~10-20%. Situasi mana yang membuatmu menerima ketidakdeterminisan demi kecepatan — dan situasi mana yang tidak?

3. Checkpoint kamu menyimpan `dirty=True/False`. Bayangkan kamu menemukan checkpoint lama dengan `dirty=True`. Apa langkah yang akan kamu ambil sebelum menggunakan hasilnya di paper?

### Jawaban Refleksi

**1. Tiga minimum dari folder eksperimen:**
> *[tulis di sini]*

**2. Kapan menerima/menolak non-determinism:**
> *[tulis di sini]*

**3. Checkpoint dengan dirty=True:**
> *[tulis di sini]*